In [18]:
from pyspark.sql import SparkSession
from pyspark.storagelevel import StorageLevel
from pyspark.sql.functions import *
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np


KHỞI TẠO SPARK

In [19]:
spark = (
    SparkSession.builder
    #.master("spark://100.77.205.48:7077")
    .appName("NYC Taxi Data")
    #.config("spark.executor.memory", "1g")
    #.config("spark.driver.memory", "1g")
    .getOrCreate()
)

#print(spark.sparkContext.master)


spark.sparkContext.setLogLevel("WARN")

spark://100.77.205.48:7077


ĐỌC DỮ LIỆU TỪ HDFS

In [20]:
#trip_path = "hdfs://master:9000/data/yellow_tripdata_2026-01.parquet"
trip_path = "hdfs://localhost:9000/data/yellow_tripdata_2026-01.parquet"

#zone_path = "hdfs://master:9000/data/taxi_zone_lookup.csv"
zone_path = "hdfs://localhost:9000/data/taxi_zone_lookup.csv"

trip_df = spark.read.parquet(trip_path)

zone_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(zone_path)
)

KIỂM TRA DỮ LIỆU BAN ĐẦU

In [21]:
print("=" * 80)
print("DATASET INFORMATION")
print("=" * 80)

print("Number of rows:", trip_df.count())
print("Number of columns:", len(trip_df.columns))

trip_df.printSchema()

DATASET INFORMATION
Number of rows: 3724889
Number of columns: 20
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



CHUẨN BỊ BẢNG ZONE LOOKUP

In [22]:
pickup_zone = zone_df.select(
    col("LocationID").alias("PULocationID"),
    col("Borough").alias("pickup_borough"),
    col("Zone").alias("pickup_zone"),
    col("service_zone").alias("pickup_service_zone")
)

dropoff_zone = zone_df.select(
    col("LocationID").alias("DOLocationID"),
    col("Borough").alias("dropoff_borough"),
    col("Zone").alias("dropoff_zone"),
    col("service_zone").alias("dropoff_service_zone")
)

JOIN DỮ LIỆU

In [23]:
taxi_df = (
    trip_df
    .join(pickup_zone, "PULocationID", "left")
    .join(dropoff_zone, "DOLocationID", "left")
)

CACHE DỮ LIỆU

In [24]:
# Cache trong RAM
taxi_df.cache()

# Kích hoạt cache
print("Rows after join:", taxi_df.count())
print("Number of columns:", len(taxi_df.columns))


Rows after join: 3724889
Number of columns: 26


KIỂM TRA EXECUTION PLAN

In [25]:
taxi_df.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(LeftOuter, [DOLocationID])
:- Project [PULocationID#14555, VendorID#14548, tpep_pickup_datetime#14549, tpep_dropoff_datetime#14550, passenger_count#14551L, trip_distance#14552, RatecodeID#14553L, store_and_fwd_flag#14554, DOLocationID#14556, payment_type#14557L, fare_amount#14558, extra#14559, mta_tax#14560, tip_amount#14561, tolls_amount#14562, improvement_surcharge#14563, total_amount#14564, congestion_surcharge#14565, Airport_fee#14566, cbd_congestion_fee#14567, pickup_borough#14614, pickup_zone#14615, pickup_service_zone#14616]
:  +- Join LeftOuter, (PULocationID#14555 = PULocationID#14613)
:     :- Relation [VendorID#14548,tpep_pickup_datetime#14549,tpep_dropoff_datetime#14550,passenger_count#14551L,trip_distance#14552,RatecodeID#14553L,store_and_fwd_flag#14554,PULocationID#14555,DOLocationID#14556,payment_type#14557L,fare_amount#14558,extra#14559,mta_tax#14560,tip_amount#14561,tolls_amount#14562,improvement_surcharge#14563,total_amount#14

EDA - TỔNG QUAN DỮ LIỆU

Shape

In [26]:
print("=" * 80)
print("DATA SHAPE")
print("=" * 80)

print("Rows:", taxi_df.count())
print("Columns:", len(taxi_df.columns))

DATA SHAPE
Rows: 3724889
Columns: 26


Xem dữ liệu mẫu

In [27]:
taxi_df.show(10, truncate=False)

+------------+------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------------+------------------------+-------------------+---------------+---------------------+--------------------+
|DOLocationID|PULocationID|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|pickup_borough|pickup_zone             |pickup_service_zone|dropoff_borough|dropoff_zone         |dropoff_service_zone|
+------------+------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+-----------+-----+-------+----

Schema

In [28]:
taxi_df.printSchema()

root
 |-- DOLocationID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- pickup_borough: string (nullable = true)
 |-- pickup_zone: string (null

EDA - MISSING VALUE

In [29]:
print("=" * 80)
print("MISSING VALUES")
print("=" * 80)

missing_df = taxi_df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in taxi_df.columns
])

missing_df.show(truncate=False)

MISSING VALUES
+------------+------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------------+-----------+-------------------+---------------+------------+--------------------+
|DOLocationID|PULocationID|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|pickup_borough|pickup_zone|pickup_service_zone|dropoff_borough|dropoff_zone|dropoff_service_zone|
+------------+------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+-----------+-----+-------+----------+------------+---------

EDA - DUPLICATE RECORDS

In [30]:
print("=" * 80)
print("DUPLICATE CHECK")
print("=" * 80)

total_rows = taxi_df.count()

distinct_rows = taxi_df.dropDuplicates().count()

duplicate_rows = total_rows - distinct_rows

print("Total rows:", total_rows)
print("Distinct rows:", distinct_rows)
print("Duplicate rows:", duplicate_rows)

DUPLICATE CHECK
Total rows: 3724889
Distinct rows: 3724889
Duplicate rows: 0


EDA - THỐNG KÊ MÔ TẢ

In [31]:
numeric_cols = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount"
]

taxi_df.select(numeric_cols).describe().show()

+-------+------------------+-----------------+------------------+------------------+-------------------+------------------+
|summary|   passenger_count|    trip_distance|       fare_amount|        tip_amount|       tolls_amount|      total_amount|
+-------+------------------+-----------------+------------------+------------------+-------------------+------------------+
|  count|           2636831|          3724889|           3724889|           3724889|            3724889|           3724889|
|   mean| 1.256271258946819|6.455646860885151|20.804253893199448|2.6081422399430054|0.49837513010477397|29.178525515798285|
| stddev|0.6702431378098668|648.8855284529166|18.927007021274658| 3.917404766075157| 2.1317308120779592|22.585529763602636|
|    min|                 0|              0.0|           -2555.2|            -88.88|              -94.5|           -2560.2|
|    max|                 9|        269097.48|            2555.2|             766.0|             122.22|            2560.2|
+-------

EDA - XEM GIÁ TRỊ TRONG CÁC BIẾN

In [32]:
categorical_cols = [
    "VendorID",
    "passenger_count",
    "RatecodeID",
    "payment_type",
    "pickup_borough",
    "dropoff_borough"
]

for c in categorical_cols:

    print("\n")
    print("="*80)
    print(f"COLUMN: {c}")
    print("="*80)

    values = (
        taxi_df.select(c)
          .distinct()
          .orderBy(c)
    )

    values.show(100, truncate=False)

    print(
        "Distinct values:",
        values.count()
    )



COLUMN: VendorID
+--------+
|VendorID|
+--------+
|1       |
|2       |
|6       |
|7       |
+--------+

Distinct values: 4


COLUMN: passenger_count
+---------------+
|passenger_count|
+---------------+
|NULL           |
|0              |
|1              |
|2              |
|3              |
|4              |
|5              |
|6              |
|7              |
|8              |
|9              |
+---------------+

Distinct values: 11


COLUMN: RatecodeID
+----------+
|RatecodeID|
+----------+
|NULL      |
|1         |
|2         |
|3         |
|4         |
|5         |
|6         |
|99        |
+----------+

Distinct values: 8


COLUMN: payment_type
+------------+
|payment_type|
+------------+
|0           |
|1           |
|2           |
|3           |
|4           |
+------------+

Distinct values: 5


COLUMN: pickup_borough
+--------------+
|pickup_borough|
+--------------+
|Bronx         |
|Brooklyn      |
|EWR           |
|Manhattan     |
|N/A           |
|Queens        |
|St

LƯU DỮ LIỆU ĐÃ JOIN

In [33]:
taxi_df.write \
    .mode("overwrite") \
    .parquet(
        #"hdfs://master:9000/output/yellow_taxi_joined"
        "hdfs://localhost:9000/output/yellow_taxi_joined"
    )

KẾT THÚC

In [34]:
print("EDA COMPLETED SUCCESSFULLY")
spark.stop()

EDA COMPLETED SUCCESSFULLY
